<a href="https://colab.research.google.com/github/SNayak-166/Evaluation_of_RecommenderSystem/blob/genreencoding/Evaluation_of_RecommenderSystem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import getpass

# 1. Securely ask the evaluator for their Kaggle credentials
if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
    print("--- Kaggle Credentials Required ---")
    os.environ["KAGGLE_USERNAME"] = input("Enter your Kaggle Username: ")
    os.environ["KAGGLE_KEY"] = getpass.getpass("Enter your Kaggle API Key (input will be hidden): ")

# 2. Install and import kagglehub
!pip install kagglehub --quiet
import kagglehub
import pandas as pd

# 3. Download the dataset
path = kagglehub.dataset_download("parasharmanas/movie-recommendation-system")
print("Dataset downloaded to:", path)

# 4. Safely load your files (example: loading movies dataset)
# kagglehub downloads a folder, so we use os.path.join to target the files inside it
ratings_df = pd.read_csv(os.path.join(path, "ratings.csv"))


--- Kaggle Credentials Required ---
Enter your Kaggle Username: Satwik.Nayak26
Enter your Kaggle API Key (input will be hidden): ··········
Using Colab cache for faster access to the 'movie-recommendation-system' dataset.
Dataset downloaded to: /kaggle/input/movie-recommendation-system


In [2]:
import pandas as pd

# Replace with your actual GitHub raw file URL
github_url = "https://raw.githubusercontent.com/SNayak-166/Evaluation_of_RecommenderSystem/refs/heads/genreencoding/processed_movies.csv"

# Load the processed movies dataframe
processed_movies_df = pd.read_csv(github_url)

# Verify the data loaded correctly
# print("Processed movies shape:", processed_movies_df.shape)
# print(processed_movies_df.head(3))

In [3]:
import numpy as np
from sklearn.model_selection import train_test_split

# Assuming rating_df and processed_movies_df are already loaded from Phase 1
# Merge ratings with processed movies
df = pd.merge(ratings_df, processed_movies_df, on='movieId')

# Create binary target for classification (Content-Based models)
# Rating >= 4.0 is considered a positive preference (1), else 0
df['liked'] = (df['rating'] >= 4.0).astype(int)

# Define genre feature columns (exclude non-genre columns)
non_feature_cols = ['userId', 'movieId', 'rating', 'timestamp', 'title', 'genres', 'genre_list', 'liked']
genre_columns = [col for col in df.columns if col not in non_feature_cols]

print(f"Total interaction records: {len(df)}")
print(f"Features used for Content-Based models: {genre_columns}")

Total interaction records: 25000095
Features used for Content-Based models: ['(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


Content based Recommender System Implementation

In [5]:
from sklearn.linear_model import SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import gc

# Initialize models
lr_sgd = SGDClassifier(loss='log_loss', penalty='l2', alpha=0.0001, random_state=42)
nb_model = MultinomialNB()

chunk_size = 150000
print("Starting multi-batch training across the dataset...")

# We will save the last chunk to act as our test set for final evaluation
X_test_chunks = []
y_test_chunks = []

for chunk_idx, chunk in enumerate(pd.read_csv('ratings.csv', chunksize=chunk_size)):

    # Merge chunk with processed movies
    chunk_merged = pd.merge(chunk, processed_movies_df, on='movieId')
    chunk_merged['liked'] = (chunk_merged['rating'] >= 4.0).astype(int)

    X_chunk = chunk_merged[genre_columns]
    y_chunk = chunk_merged['liked']

    # Let's use the last few chunks (or hold out 20% of the last chunk) for testing
    # so we can print the classification report at the end.
    X_tr, X_te, y_tr, y_te = train_test_split(X_chunk, y_chunk, test_size=0.2, random_state=42)

    # Train Naive Bayes incrementally
    if chunk_idx == 0:
        nb_model.partial_fit(X_tr, y_tr, classes=np.array([0, 1]))
    else:
        nb_model.partial_fit(X_tr, y_tr)

    # Train Logistic Regression incrementally
    lr_sgd.partial_fit(X_tr, y_tr, classes=np.array([0, 1]))

    # Accumulate test data from chunks to evaluate at the end
    X_test_chunks.append(X_te)
    y_test_chunks.append(y_te)

    if (chunk_idx + 1) % 5 == 0:
        print(f"Processed batch {chunk_idx + 1}...")

    del chunk, chunk_merged, X_chunk, y_chunk, X_tr, X_te, y_tr, y_te
    gc.collect()

# Combine accumulated test sets for final evaluation
X_test = pd.concat(X_test_chunks, ignore_index=True)
y_test = pd.concat(y_test_chunks, ignore_index=True)

# Generate predictions
nb_preds = nb_model.predict(X_test)
lr_preds = lr_sgd.predict(X_test)

# Print the final performance reports matching your desired output format
print("\n--- Naive Bayes Performance ---")
print(classification_report(y_test, nb_preds))

print("--- Logistic Regression Performance ---")
print(classification_report(y_test, lr_preds))

--- Naive Bayes Performance ---
              precision    recall  f1-score   support

           0       0.55      0.56      0.56      5006
           1       0.55      0.55      0.55      4994

    accuracy                           0.55     10000
   macro avg       0.55      0.55      0.55     10000
weighted avg       0.55      0.55      0.55     10000

--- Logistic Regression Performance ---
              precision    recall  f1-score   support

           0       0.55      0.60      0.58      5006
           1       0.56      0.51      0.53      4994

    accuracy                           0.55     10000
   macro avg       0.55      0.55      0.55     10000
weighted avg       0.55      0.55      0.55     10000



In [6]:
!pip install scikit-surprise --quiet

In [7]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split as surprise_train_test_split
from surprise import accuracy
import gc

# 1. RAM Optimization: Sample ratings_df if it has hundreds of thousands of rows
if len(ratings_df) > 100000:
    ratings_subset = ratings_df.sample(n=100000, random_state=42)
else:
    ratings_subset = ratings_df

# Prepare data for Surprise library using the subset
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings_subset[['userId', 'movieId', 'rating']], reader)

# Free up memory from raw dataframes if no longer needed immediately
del ratings_subset
gc.collect()

# Split data for Surprise
trainset, testset = surprise_train_test_split(data, test_size=0.2, random_state=42)

# Train SVD Model (Reduced n_factors from 50 to 30 for lower memory footprint)
svd_model = SVD(n_factors=30, n_epochs=15, lr_all=0.005, reg_all=0.02, random_state=42)
svd_model.fit(trainset)

# Evaluate on test set
svd_predictions = svd_model.test(testset)
print("--- SVD Evaluation ---")
accuracy.rmse(svd_predictions)
accuracy.mae(svd_predictions)

# Clean up predictions list after evaluation to free RAM
del svd_predictions
gc.collect()

--- SVD Evaluation ---
RMSE: 0.9782
MAE:  0.7571


0

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.linear_model import ElasticNet
import gc

# 1. RAM Optimization: Sample ratings if dataset is massive
if len(ratings_df) > 50000:
    ratings_subset = ratings_df.sample(n=50000, random_state=42)
else:
    ratings_subset = ratings_df

# 2. Create sparse user-item matrix directly (NO dense conversion via .toarray())
users = ratings_subset['userId'].astype('category')
movies = ratings_subset['movieId'].astype('category')

user_item_matrix = csr_matrix(
    (ratings_subset['rating'], (users.cat.codes, movies.cat.codes)),
    shape=(users.nunique(), movies.nunique())
).tocsc()  # CSC format is optimized for fast column slicing

# Clean up raw subset to save RAM
del ratings_subset
gc.collect()

# SLIM Training: Fit an ElasticNet for each item column
num_items = user_item_matrix.shape[1]
# Limit to a safe subset of items (e.g., first 200 items) for fast, crash-free execution in Colab
max_items_to_train = min(num_items, 200)

# Initialize W as a sparse or smaller dense matrix depending on need; keeping it down to trained items
W = np.zeros((max_items_to_train, num_items))

print(f"Training SLIM model on {max_items_to_train} items (Sparse Optimized)...")

for j in range(max_items_to_train):
    # Extract target column densely for regression target
    y_col = user_item_matrix[:, j].toarray().ravel()

    # Efficient column slicing for sparse matrices (excluding column j)
    left_cols = user_item_matrix[:, :j]
    right_cols = user_item_matrix[:, j+1:]

    # Concatenate sparse columns horizontally without dense conversion
    from scipy.sparse import hstack
    X_cols = hstack([left_cols, right_cols]).tocsr()

    # Train ElasticNet (using sparse input X_cols)
    en = ElasticNet(alpha=0.1, l1_ratio=0.5, positive=True, fit_intercept=False, max_iter=300)
    en.fit(X_cols, y_col)

    # Place coefficients back into weight matrix W
    coefs = en.coef_
    W[j, :j] = coefs[:j]
    W[j, j+1:] = coefs[j:]

print("SLIM Weight Matrix Shape:", W.shape)

# Run garbage collection to clean up memory
gc.collect()